# Görüntü Üretimi Uygulamaları Oluşturma (OpenAI)

LLM’ler sadece metin üretmekle kalmaz. Metin açıklamalarından görüntü de oluşturabilirsiniz. Görüntüler, MedTech, mimarlık, turizm, oyun geliştirme, pazarlama ve daha birçok alanda faydalı bir modalitedir. Bu derste, OpenAI platformundaki bugünün **GPT Görüntü** modelleriyle çalışıyoruz.

## Öğrenme hedefleri

Bu dersin sonunda şunları yapabileceksiniz:

- Görüntü üretiminin ne olduğunu ve nerelerde faydalı olduğunu açıklamak.
- `gpt-image` model ailesini anlamak ve bunun eski DALL·E modellerinden farkını kavramak.
- Bir görüntü üretim uygulaması geliştirmek ve görüntüleri düzenlemek.

## Görüntü üretimi nedir?

Görüntü üretim modelleri, bir metin isteminden görüntüler oluşturur. `gpt-image` gibi modern modeller, eğitim sırasında metin ve görüntü arasındaki ilişkiyi öğrenir, ardından rastgele gürültüyü kademeli olarak isteminize uyan bir görüntüye dönüştürür.

İki iyi bilinen görüntü modeli ailesi şunlardır:

- **`gpt-image` (OpenAI)** — bu derste kullanılan güncel nesil. Metinden görüntüye üretim ve görüntü düzenleme (maskeyle doldurma) destekler.
- **Midjourney** — kendi servisi ve Discord tabanlı iş akışı olan popüler üçüncü taraf model.

> Eski OpenAI görüntü modelleri — **DALL·E 2** ve **DALL·E 3** — eskidir. DALL·E 3 yeni dağıtımlar için artık mevcut değil ve `create_variation` gibi özellikler sadece DALL·E 2’de vardı. Yeni uygulamalar için `gpt-image` modellerini kullanın.

> **Önemli:** `gpt-image` modelleri oluşturulan görüntüyü bir **base64** (`b64_json`) olarak döndürür, URL olarak değil. Kodunuz base64 dizgesini byte’a çözer ve kaydeder — indirilecek bir görüntü URL’si yoktur.


## İlk görüntü oluşturma uygulamanızı oluşturma

Peki, bir görüntü oluşturma uygulaması oluşturmak için ne gerekir? Aşağıdaki kütüphanelere ihtiyacınız var:

- **python-dotenv**, gizlilik bilgilerinizi koddan uzak tutmak için *.env* dosyasında saklamak adına bu kütüphaneyi kullanmanız şiddetle tavsiye edilir.
- **openai**, OpenAI API ile etkileşimde bulunmak için kullanacağınız kütüphane.
- **pillow**, Python'da görüntülerle çalışmak için.


1. Aşağıdaki içeriğe sahip *.env* dosyası oluşturun:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Yukarıdaki kütüphaneleri *requirements.txt* adlı bir dosyada şu şekilde toplayın:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Sonra, sanal ortam oluşturun ve kütüphaneleri kurun:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows için sanal ortamınızı oluşturmak ve etkinleştirmek için aşağıdaki komutları kullanın:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* adlı dosyada aşağıdaki kodu ekleyin:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # OpenAI nesnesi oluştur (OPENAI_API_KEY değerini .env dosyanızdan okur)
    client = openai.OpenAI()


    try:
        # Görüntü oluşturma API'si kullanarak bir resim oluştur
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # İsteğinizi buraya yazın
            size='1024x1024',
            n=1
        )
        # Kaydedilen görüntü için dizini ayarla
        image_dir = os.path.join(os.curdir, 'images')

        # Eğer dizin yoksa oluştur
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Görüntü yolunu başlat (dosya türünün png olduğunu unutmayın)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modelleri resmi URL yerine base64 (b64_json) olarak döner
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Görüntüyü varsayılan resim görüntüleyicide göster
        image = Image.open(image_path)
        image.show()

    # Hataları yakala
    except openai.BadRequestError as err:
        print(err)

    ```

Bu kodu açıklayalım:

- Öncelikle, OpenAI kütüphanesi, dotenv kütüphanesi, base64 modülü ve Pillow kütüphanesi dahil olmak üzere ihtiyacımız olan kütüphaneleri ithal ediyoruz.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Daha sonra, API anahtarınızı ``.env`` dosyasından okuyan istemciyi oluşturuyoruz.

    ```python
    # OpenAI nesnesi oluşturun
    client = openai.OpenAI()
    ```

- Sonra, resmi oluşturuyoruz:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # İstek metninizi buraya girin
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modelleri resmi `data[0].b64_json` içinde **base64** dizgisi olarak döndürür. Bunu baytlara çözüp bir dosyaya yazıyoruz — indirilecek bir URL yoktur.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Son olarak, resmi açıp standart görüntüleyici ile gösteriyoruz:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Resim oluşturma hakkında daha fazla detay

`images.generate` parametrelerine bakalım:

- **model**, kullanılacak resim modeli (örneğin `gpt-image-1`).
- **prompt**, resmin oluşturulmasında kullanılan metin istemi. Burada "Sisli bir çayırda zambakların yetiştiği yerde, at üzerinde bir tavşan, elinde lolipop" ifadesi.
- **size**, oluşturulan resmin boyutu (örneğin `1024x1024`, `1536x1024`, `1024x1536` veya `"auto"`).
- **n**, oluşturulan resim sayısı. Burada bir tane oluşturuluyoruz.

> Resim modelleri `temperature` parametresini almaz — bu metin oluşturma kontrolüdür. Çeşitlilik için API'yi tekrar çağırın; çeşitliliği azaltmak için isteminizi daha spesifik yapın.

## Resim oluşturmanın ek yetenekleri

Birkaç satır Python ile nasıl resim oluşturduğunuzu gördünüz. `gpt-image` modelleri ayrıca mevcut resmi **düzenleyebilir**. Bir resim, isteğe bağlı bir **maske** (değiştirilecek alanı işaretler) ve bir istem sağlayarak, bir resmin bir bölümünü değiştirebilirsiniz — örneğin tavşanımıza şapka eklemek gibi.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# düzenlemeler ayrıca base64 olarak döndürülür
edited_image = base64.b64decode(response.data[0].b64_json)
```

Temel resim sadece tavşanı içerir; son resimde tavşanın üzerinde şapka vardır.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
